# Lesson 1: Agentic RAG - Code Along 6 - Using Toy AI Kit

Now, instead of implementing everything in basic Python, we'll switch to a first
framework.
For now, we'll use Toy AI Kit, which was developled for eduational purposes by
DataTalksClub in a previous workshop.
It is not meant to be used for production, but it will teach usage patterns, so
it's a great first start.

In [ ]:
from toyaikit.llm import AnthropicClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import AnthropicMessagesRunner, DisplayingRunnerCallback
from sqlitesearch import TextSearchIndex

In [2]:
# system prompt
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [3]:
# connect to db and load data
index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [4]:
# define the search function
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [5]:
# register search as a tool
agent_tools = Tools()
agent_tools.add_tool(search)

In [6]:
# see if it worked by checking the agent's available tools
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [7]:
# initialize chat interface for Jupyter notebooks
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [8]:
# initialize a tiny program (runner) for handling all the requests
runner = AnthropicMessagesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=AnthropicClient(model="claude-haiku-4-5"),
)

In [9]:
# test the runner on a question with a typo (Olama vs Ollama)
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


In [14]:
# inspect result
print(result.cost, "\n")
print(result.all_messages)

CostInfo(input_cost=Decimal('0.003812'), output_cost=Decimal('0.00256'), total_cost=Decimal('0.006372')) 

[{'role': 'system', 'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore."}, {'role': 'user', 'content': 'How do I run Olama locally?'}, {'role': 'assistant', 'content': [TextBlock(citations=None, text="I'll s

In [15]:
# continue the conversation
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


## Run an Interactive Chat

In [17]:
# start an interactive chat
runner.run()

-> Response received


-> Response received


-> Response received


-> Response received


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.8: user messages must have non-empty content'}, 'request_id': 'req_011CcGfexFYrQsU1cYMqj2hA'}

Pretty cool! The only thing that's annoying is that it errors upon quitting the
chat.
So like when I press escape, it errors just as visible above.
But still pretty impressive and cool!